## 1. Setup and Imports
This section imports all necessary libraries for data handling, model building, training, and evaluation.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from datasets import load_dataset # For Hugging Face datasets
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import datetime
from torchsummary import summary
from collections import defaultdict, OrderedDict
import warnings
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from typing import Optional, List, Dict, Tuple # Added for type hinting

# Suppress unnecessary warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Set device for training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Using device: cuda


## 2. Deepfake Dataset Class
This custom PyTorch `Dataset` class handles loading images and labels from a Hugging Face dataset. It includes options for preloading data into memory for faster access and converting grayscale images to RGB.

In [2]:
class DeepfakeDataset(Dataset):
    """Custom PyTorch Dataset for Deepfake classification"""
    
    def __init__(self, hf_dataset, transform=None, preload=False):
        """
        Args:
            hf_dataset: Hugging Face dataset object
            transform: Optional torchvision transforms
            preload: Whether to load all images into memory upfront
        """
        self.dataset = hf_dataset
        self.transform = transform
        self.preload = preload
        
        if preload:
            self.images = []
            self.labels = []
            print("Preloading dataset into memory...")
            for item in tqdm(hf_dataset, desc="Preloading"):
                self.images.append(item['image'])
                self.labels.append(item['label'])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if self.preload:
            image = self.images[idx]
            label = self.labels[idx]
        else:
            item = self.dataset[idx]
            image = item['image']
            label = item['label']
        
        # Convert grayscale to RGB if needed
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)  # Using long for classification

## 3. Data Loading and Preprocessing
This section defines functions for getting image transformations (augmentation for training, simple resizing for validation/testing) and preparing `DataLoader` objects. It also includes logic for handling class imbalance using `WeightedRandomSampler`.

In [3]:
def get_transforms(img_size=224, augment=True, color_jitter=0.2, random_erase_prob=0.1):
    """Return train and validation transforms"""
    if augment:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(
                brightness=color_jitter,
                contrast=color_jitter,
                saturation=color_jitter,
                hue=0.1
            ),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=random_erase_prob, scale=(0.02, 0.2))
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

def prepare_dataloaders(cfg):
    """Prepare train, validation, and test dataloaders from pre-split directories"""
    train_transform, val_transform = get_transforms(
        img_size=cfg.img_size,
        augment=cfg.augment,
        color_jitter=cfg.color_jitter,
        random_erase_prob=cfg.random_erase_prob
    )
    
    # Define paths for pre-split datasets
    train_dir = os.path.join(cfg.data_dir, 'train')
    val_dir = os.path.join(cfg.data_dir, 'val')
    test_dir = os.path.join(cfg.data_dir, 'test')
    
    # Load datasets using ImageFolder and apply respective transforms
    train_dataset = ImageFolder(root=train_dir, transform=train_transform)
    val_dataset = ImageFolder(root=val_dir, transform=val_transform)
    test_dataset = ImageFolder(root=test_dir, transform=val_transform)
    
    # Update class names in config based on the loaded dataset
    cfg.class_names = train_dataset.classes
    with open(os.path.join(cfg.exp_dir, "class_names.json"), 'w') as f:
        json.dump(cfg.class_names, f)
    
    # Handle class imbalance for the training set
    sampler = None
    shuffle = True
    if cfg.class_weights:
        # Get targets from the training dataset
        targets = [label for _, label in train_dataset.samples]
        class_counts = np.bincount(targets)
        class_weights_array = 1. / class_counts
        class_weights_array = class_weights_array / class_weights_array.sum()
        
        sample_weights = [class_weights_array[label] for label in targets]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        shuffle = False # Sampler handles shuffling
    
    # Create optimized dataloaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=cfg.batch_size, 
        shuffle=shuffle, 
        sampler=sampler,
        num_workers=4, 
        pin_memory=True,
        # persistent_workers=True # Enable if num_workers > 0 and you have enough memory
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=cfg.batch_size * 2, 
        num_workers=4, 
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size * 2,
        num_workers=4,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

# Example usage (for testing data loading setup)
if __name__ == '__main__':
    # Create a dummy config for testing
    class DummyConfig:
        def __init__(self):
            self.data_dir = "./dummy_dataset" # Create this directory with 'train/real', 'train/fake', etc.
            self.batch_size = 32
            self.img_size = 224
            self.val_split = 0.1 # These splits are no longer used for ImageFolder, but kept for consistency
            self.test_split = 0.1 # These splits are no longer used for ImageFolder, but kept for consistency
            self.class_weights = True
            self.augment = True
            self.color_jitter = 0.2
            self.random_erase_prob = 0.1
            self.exp_dir = "./dummy_experiments"
            self.class_names = ["Real", "Fake"]
            os.makedirs(self.exp_dir, exist_ok=True)
            
    # Create dummy data for ImageFolder to work
    os.makedirs("./dummy_dataset/train/real", exist_ok=True)
    os.makedirs("./dummy_dataset/train/fake", exist_ok=True)
    os.makedirs("./dummy_dataset/val/real", exist_ok=True)
    os.makedirs("./dummy_dataset/val/fake", exist_ok=True)
    os.makedirs("./dummy_dataset/test/real", exist_ok=True)
    os.makedirs("./dummy_dataset/test/fake", exist_ok=True)

    Image.new('RGB', (224, 224), color = 'red').save('./dummy_dataset/train/real/img1.png')
    Image.new('RGB', (224, 224), color = 'blue').save('./dummy_dataset/train/fake/img2.png')
    Image.new('RGB', (224, 224), color = 'green').save('./dummy_dataset/val/real/img3.png')
    Image.new('RGB', (224, 224), color = 'yellow').save('./dummy_dataset/val/fake/img4.png')
    Image.new('RGB', (224, 224), color = 'purple').save('./dummy_dataset/test/real/img5.png')
    Image.new('RGB', (224, 224), color = 'orange').save('./dummy_dataset/test/fake/img6.png')
    
    dummy_cfg = DummyConfig()
    print("\n📂 Loading and preparing data (dummy test)...")
    try:
        train_loader, val_loader, test_loader = prepare_dataloaders(dummy_cfg)
        sample_imgs, sample_labels = next(iter(train_loader))
        print(f"\n✅ Sample batch shape: {sample_imgs.shape}")
        print(f"✅ Sample labels: {sample_labels[:8]}")
        print(f"✅ Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    except Exception as e:
        print(f"Error during dummy data loading: {e}")
    finally:
        # Clean up dummy data
        import shutil
        shutil.rmtree('./dummy_dataset')
        shutil.rmtree('./dummy_experiments')



📂 Loading and preparing data (dummy test)...

✅ Sample batch shape: torch.Size([2, 3, 224, 224])
✅ Sample labels: tensor([0, 0])
✅ Training batches: 1, Validation batches: 1, Test batches: 1


## 4. Model Architectures
This section defines the functions to build different pre-trained CNN models (ResNet50, EfficientNetB0, DenseNet121, MobileNetV3) with custom classification heads for binary deepfake detection. Each model can be configured to freeze backbone layers, adjust dropout, and use pre-trained weights.

## Hybrid Ensemble Model
CAM Block and SE Block

In [18]:
# CBAM Block
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
            nn.Sigmoid()
        )
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.channel_attention(x)
        x = x * ca

        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.cat([avg_out, max_out], dim=1)
        sa = self.spatial_attention(sa)
        x = x * sa
        return x

# SE Block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.se(x)
        return x * w

# Hybrid Ensemble Model
class HEACNN(nn.Module):
    def __init__(self, freeze_backbones=True, dropout=0.5, pretrained=True):
        super().__init__()

        weights_rn = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = models.resnet50(weights=weights_rn)
        if freeze_backbones:
            for param in self.resnet.parameters():
                param.requires_grad = False
        self.resnet_cbam = CBAMBlock(2048)

        weights_eff = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.efficientnet = models.efficientnet_b0(weights=weights_eff)
        if freeze_backbones:
            for param in self.efficientnet.parameters():
                param.requires_grad = False

        weights_vgg = models.VGG16_Weights.DEFAULT if pretrained else None
        self.vgg = models.vgg16(weights=weights_vgg).features
        if freeze_backbones:
            for param in self.vgg.parameters():
                param.requires_grad = False
        self.vgg_pool = nn.AdaptiveAvgPool2d((7, 7))
        self.vgg_se = SEBlock(512)

        self.resnet_fc = nn.Linear(2048, 1)
        self.efficientnet_fc = nn.Linear(1000, 1)
        self.vgg_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(4096, 1)
        )

        self.fusion = nn.Sequential(
            nn.Linear(3, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        rn_feats = self.resnet.forward_features(x) if hasattr(self.resnet, 'forward_features') else self.resnet.forward(x)
        if isinstance(rn_feats, torch.Tensor) and rn_feats.dim() == 2:
            rn_feats = rn_feats.unsqueeze(-1).unsqueeze(-1)
        rn_feats = self.resnet_cbam(rn_feats)
        rn_feats = torch.flatten(nn.AdaptiveAvgPool2d(1)(rn_feats), 1)
        rn_out = self.resnet_fc(rn_feats)

        eff_out = self.efficientnet(x)
        eff_out = self.efficientnet_fc(eff_out)

        vgg_feats = self.vgg(x)
        vgg_feats = self.vgg_se(vgg_feats)
        vgg_feats = self.vgg_pool(vgg_feats)
        vgg_out = self.vgg_fc(vgg_feats)

        logits = torch.cat([rn_out, eff_out, vgg_out], dim=1)
        output = self.fusion(logits)
        return output

def build_HEACNN(freeze=True, dropout=0.4, pretrained=True):
    """Build configured DenseNet121Binary model"""
    return HEACNN( dropout=dropout, pretrained=pretrained)

## 5. Training Loop
This section implements the core training logic, including forward and backward passes, optimizer steps, learning rate scheduling, mixed precision training, early stopping, and model checkpointing. It also logs metrics to TensorBoard for visualization.

In [19]:
def train_model(model, train_loader, val_loader, cfg):
    """Training loop with early stopping, learning rate scheduling, and mixed precision"""
    # Setup device
    device = next(model.parameters()).device
    
    # Setup optimization
    if cfg.optimizer == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    
    # Setup learning rate scheduler
    if cfg.scheduler == "ReduceLROnPlateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=cfg.patience // 2, min_lr=cfg.min_lr
        )
    else: # CosineAnnealingWarmRestarts
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=cfg.epochs // 5, T_mult=1, eta_min=cfg.min_lr
        )
    
    # Loss function (BCEWithLogitsLoss combines sigmoid and BCELoss for numerical stability)
    criterion = nn.BCEWithLogitsLoss()
    
    # Mixed precision training scaler
    scaler = GradScaler(enabled=cfg.mixed_precision)
    
    # Initialize tracking
    history = defaultdict(list)
    best_metrics = {'auc': 0.0, 'epoch': 0, 'val_loss': float('inf')}
    early_stop_counter = 0
    writer = SummaryWriter(cfg.log_dir)
    
    # Training loop
    for epoch in range(cfg.epochs):
        # Unfreeze backbone after certain epochs
        if cfg.freeze_backbone and epoch == cfg.freeze_epochs:
            print("\nUnfreezing backbone layers...")
            for param in model.parameters():
                param.requires_grad = True
            # Re-initialize optimizer with all parameters now trainable
            if cfg.optimizer == "AdamW":
                optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            else:
                optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            # Re-initialize scheduler with new optimizer
            if cfg.scheduler == "ReduceLROnPlateau":
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='max', factor=0.5, patience=cfg.patience // 2, min_lr=cfg.min_lr
                )
            else:
                scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                    optimizer, T_0=cfg.epochs // 5, T_mult=1, eta_min=cfg.min_lr
                )
        
        # Training phase
        model.train()
        running_loss = 0.0
        train_metrics = defaultdict(float)
        
        for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")):
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Mixed precision forward pass
            with autocast(enabled=cfg.mixed_precision):
                preds = model(x)
                loss = criterion(preds, y)
            
            # Backward pass and optimizer step with scaler
            scaler.scale(loss).backward()
            
            # Gradient clipping (optional, but good practice with mixed precision)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            # Update metrics
            running_loss += loss.item() * x.size(0)
            y_pred = (preds > 0).int() # Convert logits to binary predictions
            train_metrics['acc'] += accuracy_score(y.cpu().numpy(), y_pred.cpu().numpy()) * x.size(0)
        
        # Calculate epoch metrics
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = train_metrics['acc'] / len(train_loader.dataset)
        
        # Validation phase
        val_metrics, _, _, _ = evaluate_model(model, val_loader, criterion, device)
        
        # Learning rate scheduling
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_metrics['auc'])
        else:
            scheduler.step()
        
        # Update history
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_f1'].append(val_metrics['f1'])
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # TensorBoard logging
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_metrics['loss'], epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
        writer.add_scalar('AUC/val', val_metrics['auc'], epoch)
        writer.add_scalar('F1/val', val_metrics['f1'], epoch)
        writer.add_scalar('Precision/val', val_metrics['precision'], epoch)
        writer.add_scalar('Recall/val', val_metrics['recall'], epoch)
        writer.add_scalar('AP/val', val_metrics['ap'], epoch)
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{cfg.epochs}:")
        print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.4f}")
        print(f"  AUC: {val_metrics['auc']:.4f} | F1: {val_metrics['f1']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f}")
        print(f"  AP: {val_metrics['ap']:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Save best model
        if val_metrics['auc'] > best_metrics['auc']:
            best_metrics = val_metrics.copy()
            best_metrics['epoch'] = epoch + 1
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_metrics': val_metrics
            }, cfg.model_path)
            print("✅ New best model saved!")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
        
        # Early stopping
        if cfg.early_stop and early_stop_counter >= cfg.patience:
            print(f"⏹ Early stopping triggered at epoch {epoch+1}")
            break

    # Save final model (last epoch's state)
    torch.save({
        'epoch': epoch + 1, # Use the last epoch reached
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_metrics': val_metrics # Last validation metrics
    }, cfg.last_model_path)
    
    # Save training history
    with open(os.path.join(cfg.exp_dir, "training_history.json"), 'w') as f:
        json.dump(history, f, indent=2)
    
    writer.close()
    return history, best_metrics

## 6. Evaluation and Visualization
This section provides functions to evaluate the trained model on validation and test sets, calculate various classification metrics, and generate insightful visualizations like confusion matrices, precision-recall curves, and training history plots.

In [20]:
def evaluate_model(model, loader, criterion=None, device=None):
    """Evaluate model performance and return metrics"""
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    y_true, y_scores, y_pred = [], [], []
    total_loss = 0.0
    
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating", leave=False):
            x, y = x.to(device), y.float().unsqueeze(1).to(device)
            
            with autocast(): # Use autocast for evaluation too if mixed precision was used in training
                preds = model(x)
                if criterion:
                    loss = criterion(preds, y)
                    total_loss += loss.item() * x.size(0)
            
            y_true.extend(y.cpu().numpy())
            y_scores.extend(torch.sigmoid(preds).cpu().numpy()) # Apply sigmoid to logits for scores
            y_pred.extend((preds > 0).int().cpu().numpy()) # Convert logits to binary predictions
    
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    y_pred = np.array(y_pred)
    
    metrics = {
        'loss': total_loss / len(loader.dataset) if criterion else 0.0,
        'accuracy': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_scores),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'ap': average_precision_score(y_true, y_scores)
    }
    
    return metrics, y_true, y_scores, y_pred

def generate_evaluation_report(model, loader, cfg, phase="val"):
    """Generate comprehensive evaluation report with visualizations"""
    metrics, y_true, y_scores, y_pred = evaluate_model(model, loader, device=next(model.parameters()).device)
    
    report = {
        'metrics': metrics,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(
            y_true, y_pred, target_names=cfg.class_names, output_dict=True
        )
    }
    
    # Save report
    report_path = os.path.join(cfg.exp_dir, f"{phase}_report.json")
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    # Plot confusion matrix
    plt.figure(figsize=(6, 6))
    sns.heatmap(report['confusion_matrix'], annot=True, fmt='d', cmap='Blues', 
                xticklabels=cfg.class_names, yticklabels=cfg.class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f"Confusion Matrix ({phase.capitalize()})")
    plt.savefig(os.path.join(cfg.exp_dir, f"{phase}_confusion_matrix.png"))
    plt.close()
    
    # Plot PR curve
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    plt.figure(figsize=(6, 4))
    plt.plot(recall, precision, label=f"AP = {metrics['ap']:.2f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve ({phase.capitalize()})")
    plt.legend()
    plt.savefig(os.path.join(cfg.exp_dir, f"{phase}_pr_curve.png"))
    plt.close()

In [21]:
def plot_training_history(history, cfg):
    """Plot training metrics and save to experiment directory"""
    plt.figure(figsize=(18, 12))
    
    # Plot loss
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot accuracy
    plt.subplot(2, 2, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Validation')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # Plot AUC
    plt.subplot(2, 2, 3)
    plt.plot(history['val_auc'], label='Validation')
    plt.title('Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    
    # Plot learning rate
    plt.subplot(2, 2, 4)
    plt.plot(history['lr'])
    plt.title('Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.exp_dir, "training_metrics.png"))
    plt.close()

## 7. Configuration and Main Execution
This section defines the `Config` class to manage all hyperparameters and paths for the experiment. The main execution block orchestrates the entire process: initializing configuration, preparing data, building and training the model, plotting results, and evaluating on validation and test sets.

In [22]:
class Config:
    def __init__(self):
        # Data configuration
        self.data_dir = "dataset" # Path to your dataset (e.g., 'data/deepfake_faces')
        self.batch_size = 64  # Optimized for modern GPUs
        self.img_size = 256   # Better resolution for detection
        self.val_split = 0.15 # These splits are now indicative, actual split is by folders
        self.test_split = 0.15 # These splits are now indicative, actual split is by folders
        self.class_weights = True  # Handle class imbalance
        self.class_names = ["Real", "Fake"] # Default class names (will be updated by DataLoader)
        
        # Training configuration
        self.epochs = 30
        self.lr = 3e-4       # Optimized learning rate
        self.min_lr = 1e-6     # Minimum learning rate
        self.weight_decay = 1e-4  # Better regularization
        self.freeze_backbone = True
        self.freeze_epochs = 5  # Freeze initial layers
        self.dropout = 0.4     # Better regularization
        
        # Augmentation configuration
        self.augment = True
        self.color_jitter = 0.3
        self.random_erase_prob = 0.2
        
        # Optimization configuration
        self.optimizer = "AdamW"  # Best for this task
        self.scheduler = "CosineAnnealingWarmRestarts"  # Better learning rate adaptation
        self.patience = 5
        self.early_stop = False
        self.mixed_precision = True  # Faster training
        
        # Model checkpointing
        self.save_top_k = 3  # Save multiple checkpoints (not fully implemented in train_model, but good to have)
        
        # Experiment tracking
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_name = f"mobilenetv3_{timestamp}" # Change model name as needed
        self.exp_dir = os.path.join("experiments", self.exp_name)
        os.makedirs(self.exp_dir, exist_ok=True)
        
        # Path configurations
        self.model_path = os.path.join(self.exp_dir, "best_model.pth")
        self.last_model_path = os.path.join(self.exp_dir, "last_model.pth")
        self.log_dir = os.path.join(self.exp_dir, "logs")
        
    def save(self):
        """Save configuration to experiment directory"""
        config_dict = {k:v for k,v in vars(self).items() if not k.startswith('__') and k != 'class_names'}
        with open(os.path.join(self.exp_dir, "config.json"), 'w') as f:
            json.dump(config_dict, f, indent=2)
            
    def __str__(self):
        return json.dumps(vars(self), indent=2)

if __name__ == "__main__":
    # Initialize configuration
    cfg = Config()
    cfg.save()
    print(f"\n⚙️ Configuration:\n{cfg}")
    
    # Prepare data
    print("\n📂 Loading and preparing data...")
    train_loader, val_loader, test_loader = prepare_dataloaders(cfg)
    
    # Build model (Choose one of the models to train)
    print("\n🧠 Building model...")
    # Example: HEACNN
    model = build_HEACNN(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    # Example: ResNet50
    # model = build_resnet50(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    # Example: EfficientNetB0
    # model = build_efficientnetb0(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    # Example: DenseNet121
    # model = build_densenet121(freeze=cfg.freeze_backbone, dropout=cfg.dropout)
    
    print(f"Model architecture:\n{model}")
    # Optional: Print model summary
    try:
        # summary(model, input_size=(3, cfg.img_size, cfg.img_size), device=device.type) # Commented out to avoid 'list' object has no attribute 'size' error
        pass # Placeholder for commented out summary call
    except Exception as e:
        print(f"Could not print model summary: {e}")
    
    # Train model
    print("\n🏋️ Starting training...")
    history, best_metrics = train_model(model, train_loader, val_loader, cfg)
    
    # Plot training history
    plot_training_history(history, cfg)
    
    # Evaluate best model on validation set
    print("\n🔍 Evaluating best model on validation set...")
    checkpoint = torch.load(cfg.model_path, weights_only=False) # Added weights_only=False
    model.load_state_dict(checkpoint['model_state_dict'])
    val_report = generate_evaluation_report(model, val_loader, cfg, "val")
    
    # Evaluate on test set
    print("\n🧪 Evaluating on test set...")
    test_report = generate_evaluation_report(model, test_loader, cfg, "test")
    
    print(f"\n🎉 Training complete! Best validation AUC: {best_metrics['auc']:.4f}")
    print(f"📁 Results saved in: {cfg.exp_dir}")


⚙️ Configuration:
{
  "data_dir": "dataset",
  "batch_size": 64,
  "img_size": 256,
  "val_split": 0.15,
  "test_split": 0.15,
  "class_weights": true,
  "class_names": [
    "Real",
    "Fake"
  ],
  "epochs": 30,
  "lr": 0.0003,
  "min_lr": 1e-06,
  "weight_decay": 0.0001,
  "freeze_backbone": true,
  "freeze_epochs": 5,
  "dropout": 0.4,
  "augment": true,
  "color_jitter": 0.3,
  "random_erase_prob": 0.2,
  "optimizer": "AdamW",
  "scheduler": "CosineAnnealingWarmRestarts",
  "patience": 5,
  "early_stop": false,
  "mixed_precision": true,
  "save_top_k": 3,
  "exp_name": "mobilenetv3_20250708_001705",
  "exp_dir": "experiments\\mobilenetv3_20250708_001705",
  "model_path": "experiments\\mobilenetv3_20250708_001705\\best_model.pth",
  "last_model_path": "experiments\\mobilenetv3_20250708_001705\\last_model.pth",
  "log_dir": "experiments\\mobilenetv3_20250708_001705\\logs"
}

📂 Loading and preparing data...

🧠 Building model...
Downloading: "https://download.pytorch.org/models/vgg

100%|███████████████████████████████████████████████████████████████████████████████| 528M/528M [02:05<00:00, 4.42MB/s]
C:\Users\Shohan\AppData\Local\Temp\ipykernel_1200\474825013.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.mixed_precision)


Model architecture:
HEACNN(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequ

C:\Users\Shohan\AppData\Local\Temp\ipykernel_1200\474825013.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
Epoch 1/30:   0%|                                                                              | 0/402 [00:23<?, ?it/s]


RuntimeError: Given groups=1, weight of size [128, 2048, 1, 1], expected input[64, 1000, 1, 1] to have 2048 channels, but got 1000 channels instead